# Week 18: Autonomous AI Agents & Multi-Agent Systems
## Notebook 04: Code Execution Engines, AST Validation & Error Reflection Loops

### 1. Conceptual Intuition & Real-World Use Case
Code agents (e.g. OpenAI Code Interpreter, SWE-bench solvers) do not just output text:
1. They generate executable Python code.
2. They validate syntax via **Abstract Syntax Tree (AST)** parsing.
3. They execute code within a secure isolated sandbox with timeout constraints.
4. If an execution error occurs (Traceback), they perform **Self-Reflection**, feeding the error back into the model to self-correct automatically.

---

### 2. Mathematical Foundations: Pass@k Metric
Evaluating code generation capabilities uses the **Pass@k** metric:
$$\text{Pass}@k = \mathbb{E}_{\text{problems}} \left[ 1 - \frac{\binom{n - c}{k}}{\binom{n}{k}} \right]$$
Where:
- $n$: Total candidate code samples generated per problem ($n \ge k$).
- $c$: Number of correct samples passing all unit test assertions.
- $k$: Evaluation sample budget (e.g. Pass@1 or Pass@10).

In [ ]:
import ast
import contextlib
import io
import traceback

class SandboxedPythonRunner:
    def __init__(self, timeout_seconds=2):
        self.timeout = timeout_seconds

    def validate_ast(self, code_string):
        """Check syntax without executing."""
        try:
            ast.parse(code_string)
            return True, "Syntax Valid"
        except SyntaxError as e:
            return False, f"Syntax Error: {e.msg} at line {e.lineno}"

    def execute(self, code_string):
        """Run code in isolated namespace and capture stdout / exceptions."""
        is_valid, msg = self.validate_ast(code_string)
        if not is_valid:
            return {"success": False, "stdout": "", "error": msg}

        # Safe restricted namespace
        local_scope = {}
        stdout_capture = io.StringIO()

        try:
            with contextlib.redirect_stdout(stdout_capture):
                # Execute compiled AST
                exec(code_string, {}, local_scope)
            return {
                "success": True,
                "stdout": stdout_capture.getvalue(),
                "error": None,
                "scope": local_scope
            }
        except Exception as e:
            return {
                "success": False,
                "stdout": stdout_capture.getvalue(),
                "error": traceback.format_exc()
            }

# Step 2: Simulate Autonomous Error Reflection Loop
runner = SandboxedPythonRunner()

# Candidate 1: Buggy code (IndexError)
buggy_code = """
def get_second_item(lst):
    return lst[1]

# Test assertion
assert get_second_item([42]) == 42
"""

print("--- Iteration 1: Executing Initial AI Code ---")
result1 = runner.execute(buggy_code)
print(f"Success: {result1['success']}")
if not result1['success']:
    print("Error Captured:\n", result1['error'].strip().splitlines()[-1])

# Self-Reflection: Agent inspects error and produces patched code
fixed_code = """
def get_second_item(lst):
    if len(lst) < 2:
        return lst[0] if lst else None
    return lst[1]

# Test assertion
assert get_second_item([42]) == 42
assert get_second_item([10, 20]) == 20
print('All test assertions passed successfully!')
"""

print("\n--- Iteration 2: Executing Reflected & Patched Code ---")
result2 = runner.execute(fixed_code)
print(f"Success: {result2['success']}")
print(f"Stdout Output: {result2['stdout'].strip()}")

### 3. Step-by-Step Code Walkthrough
1. **AST Pre-Check:** Catches syntax errors immediately without entering the execution environment.
2. **Stdout Redirection:** Captures runtime output safely in memory buffers.
3. **Reflection Feedback:** The system captures runtime exceptions and supplies them back to the prompting loop, demonstrating self-correction.

---

### 4. Industry Applications & Trade-offs
- **Data Analysis Agents:** Automatically querying CSV files, plotting graphs, and iterating until data visualization matches requirements.
- **Security Boundaries:** Real-world enterprise runners use gVisor, Docker containers, or Firecracker microVMs to block unauthorized system calls.

---

### 5. Top Interview Questions & Answers
**Q1: What are the security risks of using `exec()` in Python, and how are they prevented?**
> **Answer:** `exec()` can run malicious code like `import os; os.system('rm -rf /')` or initiate reverse shell network calls. Production environments prevent this via container isolation (Docker/Firecracker), non-root users, disabling network interfaces, and using AST visitors to forbid dangerous modules (`os`, `subprocess`, `sys`).

**Q2: What is the Pass@k unbiased estimator?**
> **Answer:** Naively estimating Pass@k by picking $k$ samples and averaging success has high variance. Chen et al. (HumanEval) derived an unbiased estimator that generates $n \ge k$ samples ($n \approx 100$) and computes expected probability using hypergeometric combinations, preventing sampling noise.